In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.diagnostic as sms
load_dotenv()


ruta_absoluta = os.getenv('RUTA_ABSOLUTA')
ruta_excel = f'{ruta_absoluta}/data/processed/BaseDatos_Vial_Huanuco.xlsx'
SHEET = 'BASE_DATOS'
# cargar datos
df = pd.read_excel(ruta_excel, sheet_name=SHEET, header=2)
y = df['ln(Y)']
x = df[[
    'ln(X1)',
    'X2 — Gasto\nPúblico Transporte\npc (S/ ejec.)',
    'ln(X5)'
]]
X = sm.add_constant(x)
modelo = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': df['PROVINCIA']})

print(modelo.summary())

residuos = modelo.resid

# autocorrelacion
dw = sm.stats.stattools.durbin_watson(residuos)
print(dw)
test_bg = sms.acorr_breusch_godfrey(modelo, nlags=2)
print(test_bg[1])


# heterocedasticidad
wh_test = sms.het_white(residuos, X)
print(wh_test[1])



                            OLS Regression Results                            
Dep. Variable:                  ln(Y)   R-squared:                       0.986
Model:                            OLS   Adj. R-squared:                  0.985
Method:                 Least Squares   F-statistic:                     412.7
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           8.91e-11
Time:                        11:50:00   Log-Likelihood:                 245.29
No. Observations:                 143   AIC:                            -482.6
Df Residuals:                     139   BIC:                            -470.7
Df Model:                           3                                         
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [2]:
modelo_mx = sm.MixedLM(y, X, groups=df['PROVINCIA']).fit()

modelo_mx.summary()

residuos_mx = modelo_mx.resid
# autocorrelacion
dw = sm.stats.stattools.durbin_watson(residuos_mx)
print(dw)
test_bg = sms.acorr_breusch_godfrey(modelo_mx, nlags=2)
print(test_bg[1])


# heterocedasticidad
wh_test = sms.het_white(residuos, X)
print(wh_test[1])

0.2351426002155017
6.552754869960056e-24
1.1776752074285432e-08
